In [2]:
!pip install -q langchain langchain-community langchain-core langchain-text-splitters
!pip install -q sentence-transformers faiss-cpu pypdf
!pip install -q langchain-huggingface
!pip install -q groq
!pip install -q gradio

In [12]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


In [4]:
from langchain_community.document_loaders import PyPDFLoader

from google.colab import files
import os

uploaded = files.upload()

pdf_files = [
    f for f in uploaded.keys()
    if f.lower().endswith(".pdf")
]

if not pdf_files:
    raise ValueError("Please upload a PDF file.")

PDF_PATH = os.path.join("/content", pdf_files[0])

print("PDF loaded:", PDF_PATH)

loader = PyPDFLoader(PDF_PATH)
documents = loader.load()

/tmp/ipykernel_2360/3033925371.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Saving Bangladesh_The_Penal_Code_1860.pdf to Bangladesh_The_Penal_Code_1860.pdf
PDF loaded: /content/Bangladesh_The_Penal_Code_1860.pdf


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

docs = text_splitter.split_documents(documents)

In [8]:
from huggingface_hub import login
from langchain_huggingface import HuggingFaceEmbeddings

hf_token = getpass("Enter your Hugging Face access token: ")
login(token=hf_token)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Hugging Face login successful. Embeddings model initialized.")

Enter your Hugging Face access token: ··········


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Hugging Face login successful. Embeddings model initialized.


In [9]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(docs, embeddings)

In [10]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [16]:
from groq import Groq
import os

client = Groq(api_key=os.environ["GROQ_API_KEY"])

def groq_llm(prompt):
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.2,
        max_completion_tokens=1024
    )

    return response.choices[0].message.content

In [17]:
def rag_chat(query):
    # Retrieve relevant legal documents
    retrieved_docs = retriever.invoke(query)

    context = "\n\n".join([
        doc.page_content for doc in retrieved_docs
    ])

    prompt = f"""
    You are an AI assistant specialized in providing legal information
    based on the uploaded legal documents.

    You have two modes:

    1. DOCUMENT MODE:
    - If the question is related to the provided legal information,
      answer using ONLY the retrieved document passages.
    - Do NOT mention the word "context" or use phrases like
      "according to the context".
    - Answer clearly, naturally, and in simple language.
    - Explain legal terms when necessary, without changing their meaning.
    - Do not invent laws, legal sections, penalties, case details,
      or legal provisions that are not present in the retrieved passages.
    - If the answer is not found, politely say:
      "The uploaded legal document does not contain sufficient
      information to answer this question."
    - When possible, mention the relevant legal section or provision
      stated in the document.
    - Ask if the user would like to know anything else about the
      legal information covered by the document.

    2. CHAT MODE:
    - If the user is greeting or making general conversation,
      respond politely and naturally.
    - You may answer simple conversational messages without using
      the legal document.
    - For unrelated legal questions, do not make unsupported claims.
      Explain that you can answer questions based on the uploaded
      legal document.
    - Ask if the user would like to know anything about the legal
      information covered by the document.

    3. LEGAL SAFETY:
    - Provide general legal information, not personalized legal advice.
    - Do not claim that a law is currently in force or has not been
      amended unless the provided document establishes this.
    - If the document is unclear or insufficient, acknowledge this
      rather than guessing.
    - Encourage users to consult a qualified legal professional for
      advice about their specific circumstances.

    ---

    LEGAL DOCUMENT PASSAGES:
    {context}

    USER QUESTION:
    {query}

    FINAL ANSWER:
    """

    answer = groq_llm(prompt)
    return answer

In [18]:
import gradio as gr

def chat_interface(message, history):
    try:
        response = rag_chat(message)
        return response
    except Exception as e:
        return f"An error occurred: {str(e)}"

demo = gr.ChatInterface(
    fn=chat_interface,
    title="⚖️ Legal Information Chatbot",
    description=(
        "Ask questions about the uploaded legal document. "
        "Answers are based on information retrieved from the document. "
        "This chatbot provides general legal information, not legal advice."
    ),
    examples=[
        "What is the purpose of this legal document?",
        "Explain the provisions related to theft.",
        "What penalties are described in the document?"
    ],
    chatbot=gr.Chatbot(height=450),
    textbox=gr.Textbox(
        placeholder="Ask a legal question...",
        lines=2
    )
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://65421743cb15ff3f97.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://65421743cb15ff3f97.gradio.live
